# Day 3 · 3교시 [주석본] ML 파이프라인 — 한 줄씩 뜯어보기

`03_ml_pipeline.ipynb` 와 **코드는 똑같고**, 설명과 줄별 주석을 붙인 학습본이다.
**ML 이 처음인 사람** 기준으로 썼다(딥러닝만 해 본 경우 포함).

---

## 0. 오늘 뭘 하는 건가

**두 가지 문제를 풀고, 나온 점수를 읽는 법을 배운다.**

| | 회귀(regression) | 분류(classification) |
|---|---|---|
| 맞히는 것 | **숫자** | **종류** |
| 오늘 데이터 | `load_diabetes` — 환자 442명의 검사 수치 10개로 **1년 뒤 당뇨 진행도**(숫자)를 예측 | `load_iris` — 붓꽃 150송이의 꽃잎·꽃받침 치수 4개로 **품종 3종**을 구분 |
| 성적표 | RMSE · R² | 정확도 · 혼동행렬 |

두 데이터 다 sklearn 에 들어 있어 **다운로드가 필요 없다.**

### 왜 이렇게 작은 데이터를 쓰나

442개, 150개다. 딥러닝을 해 봤다면 "이게 데이터셋?" 싶을 것이다.
**표(table) 형태의 작은 데이터**에서는 딥러닝보다 이런 단순한 모델이 더 잘 맞고,
빠르고, 왜 그렇게 판단했는지 설명도 된다. 실무에서 만나는 데이터의 상당수가 이렇다.

### 딥러닝과 다른 점 세 가지

**① 학습 루프가 안 보인다.** `for epoch`, `loss.backward()`, `optimizer.step()` 이
없다. **`.fit(X, y)` 한 줄이 그 전부를 대신한다.** 안에서 똑같이 손실을 줄이는 최적화가
돌지만 감춰져 있다.

**② 특성을 모델이 만들지 않는다.** 딥러닝은 raw 픽셀·텍스트에서 중간층이 특징을 뽑아낸다.
여기서는 **사람이 이미 뽑아 놓았다** — 검사 수치 10개, 꽃 치수 4개가 곧 특성이다.

**③ 모델이 아주 작다.** 오늘 쓰는 로지스틱 회귀는 딥러닝 용어로 **은닉층 0개**
(`Linear → softmax`)다. 파라미터가 수십 개 수준이다.

> 의존성: `pip install scikit-learn numpy` · 전부 오프라인 · 시드 고정

## 1. 회귀 — 숫자를 맞히는 문제 (교안 3.4)

**당뇨 진행도**를 예측한다. 정답이 `151.0`, `75.0` 같은 **연속된 숫자**다.
그래서 "맞았다/틀렸다"가 아니라 **"얼마나 빗나갔나"** 로 채점한다.

### 두 지표

- **RMSE**: 평균적으로 얼마나 빗나갔는지. **정답과 같은 단위**라 해석이 직관적이다.
  (RMSE 53 = "평균 53만큼 틀린다")
- **R²**: 0~1 사이 점수. `1.0`=완벽, **`0.0`=아무것도 안 배우고 평균만 답하는 수준**.

In [ ]:
import numpy as np                                    # 수치 계산
from sklearn.datasets import load_diabetes              # 예제 데이터 (내장)
from sklearn.linear_model import LinearRegression       # 선형회귀 = 숫자를 맞히는 가장 기본 모델
from sklearn.metrics import mean_squared_error, r2_score  # 채점 도구
from sklearn.model_selection import train_test_split    # 데이터를 train/test 로 쪼개는 함수

dia = load_diabetes()                                   # 데이터 로드 (다운로드 없음)
X, y = dia.data, dia.target                             # X=검사수치 10개, y=당뇨 진행도(숫자)
print("데이터:", X.shape, "| 정답(당뇨 진행도) 범위:", f"{y.min():.0f} ~ {y.max():.0f}")

# test_size=0.3 → 30%를 test 로 떼어 둔다. random_state 는 '무작위 고정' 값
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42)
model = LinearRegression().fit(Xtr, ytr)                # 학습 (train 만 사용)
pred = model.predict(Xte)                               # test 에 대한 예측값

# mean_squared_error 는 오차의 제곱 평균 → ** 0.5 로 제곱근을 씌우면 RMSE
rmse = mean_squared_error(yte, pred) ** 0.5
print(f"\nRMSE : {rmse:.1f}   ← 평균 이만큼 빗나간다 (정답과 같은 단위)")
print(f"R2   : {r2_score(yte, pred):.3f}   ← 1.0 이 완벽, 0.0 은 '평균만 찍는 수준'")

# 비교 기준선(baseline): 학습 없이 'train 의 평균'을 모든 답으로 내놓았다면?
baseline = np.full(len(yte), ytr.mean())                # test 개수만큼 평균값으로 채운 배열
print(f"\n비교) 아무것도 안 배우고 '평균'만 답했을 때 RMSE: "
      f"{mean_squared_error(yte, baseline) ** 0.5:.1f}")

### 결과 읽는 법

```
RMSE : 53.1
R2   : 0.477
비교) 평균만 답했을 때 RMSE: 73.7
```

정답 범위가 `25 ~ 346` 인데 평균 **53만큼** 빗나간다 — 꽤 큰 오차다.

**마지막 줄이 핵심이다.** 아무것도 학습하지 않고 "그냥 평균"이라고만 답해도 `73.7` 이다.
우리 모델은 `53.1` 이니 **평균 찍기보다 28% 덜 빗나간 정도**다.
`R² 0.477` 이라는 숫자가 바로 이 비교를 요약한 것이다.

> 🔥 **점수 하나만 보면 잘한 건지 알 수 없다.** 반드시 **"아무것도 안 했을 때"와 비교**해야
> 의미가 생긴다. 이 기준선을 **baseline** 이라 하고, 어떤 ML 문제에서든 제일 먼저 잡는다.

## 2. 분류 — 종류를 맞히는 문제 (교안 3.6)

붓꽃 품종 3가지를 구분한다. 정답이 `0/1/2` 같은 **범주**다.

### 혼동행렬이 왜 필요한가

정확도 `0.933` 은 "45개 중 42개 맞혔다"만 알려준다.
**무엇을 무엇으로 착각했는지**는 안 알려준다. 그걸 보는 표가 **혼동행렬**이다.
- **행 = 실제 정답**, **열 = 모델의 예측**
- **대각선** = 맞힌 것, **대각선 밖** = 틀린 것

In [ ]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression     # 이름은 '회귀'지만 분류 모델이다
from sklearn.metrics import accuracy_score, confusion_matrix

iris = load_iris()
Xi, yi = iris.data, iris.target                         # Xi=꽃 치수 4개, yi=품종(0/1/2)
names = [str(n) for n in iris.target_names]             # 품종 이름 3개
print("데이터:", Xi.shape, "| 품종:", names)

# stratify=yi → 품종 비율을 train/test 양쪽에 똑같이 유지 (작은 데이터에선 거의 필수)
Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(
    Xi, yi, test_size=0.3, random_state=42, stratify=yi
)
# max_iter=200: 내부 최적화 반복 상한. 기본값으로는 수렴 경고가 날 수 있어 늘려 준다
clf = LogisticRegression(max_iter=200).fit(Xi_tr, yi_tr)
ip = clf.predict(Xi_te)                                 # 예측 결과 (0/1/2 배열)
print(f"\n정확도: {accuracy_score(yi_te, ip):.3f}")

cm = confusion_matrix(yi_te, ip)                        # (3,3) 행렬: cm[실제][예측] = 개수
w = max(len(n) for n in names)                          # 출력 정렬용 폭
print("\n혼동행렬 (행=실제, 열=예측):")
print(" " * (w + 2) + "  ".join(f"{n:>{w}}" for n in names))   # 헤더 줄
for i, row in enumerate(cm):                            # 행렬을 한 줄씩 출력
    print(f"{names[i]:>{w}}  " + "  ".join(f"{v:>{w}}" for v in row))

### 결과 읽는 법

```
                setosa  versicolor   virginica
    setosa          15           0           0
versicolor           0          14           1
 virginica           0           2          13
```

- **`setosa` 행**: 15개 **전부 맞혔다.** 이 품종은 확실히 구분된다.
- **`versicolor` 행의 `1`**: 실제로는 versicolor 인데 **virginica 로 착각한 1건**
- **`virginica` 행의 `2`**: 실제로는 virginica 인데 **versicolor 로 착각한 2건**

> 🔥 **틀린 3건이 전부 `versicolor ↔ virginica` 사이에서 났다.** 우연이 아니라
> **그 두 품종이 실제로 비슷하다**는 뜻이다. 성능을 올리고 싶다면 setosa 를 더 잘 맞히려
> 애쓸 게 아니라 **저 둘을 가르는 특성**을 찾아야 한다 — 정확도 숫자만 봤다면 몰랐을 정보다.

## 3. 🔥 과적합 — train 점수가 높은 게 좋은 모델이 아니다 (교안 3.7)

지금까지 데이터는 **깨끗했다.** 실무는 그렇지 않다 — 사람이 붙인 라벨에는 **오류가 섞인다**
(잘못 분류된 불량품, 오진, 오타). 붓꽃 라벨 일부를 일부러 어지럽혀 놓고 실험한다.

### 결정 트리(decision tree)란

"꽃잎 길이 > 2.5 이면 왼쪽, 아니면 오른쪽..." 식의 **스무고개**로 분류하는 모델이다.
`max_depth` 는 **몇 번까지 물어볼지**를 정한다.
- 깊게(`None` = 무제한) → 계속 쪼개서 **데이터를 통째로 외울 수 있다**
- 얕게(`2`, `3`) → 큰 규칙만 배운다

> 🧠 딥러닝의 "층을 몇 개 쌓을까"와 같은 성격의 손잡이다. 역전파는 쓰지 않는다.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

rng = np.random.default_rng(0)                   # 난수 생성기 (시드 0 → 항상 같은 결과)
y_dirty = yi.copy()                              # 원본 라벨을 복사해서
flip = rng.choice(150, size=25, replace=False)   # 150개 중 25개 위치를 무작위로 고르고
y_dirty[flip] = rng.integers(0, 3, size=25)      # 그 자리의 라벨을 아무 값으로 바꾼다
# (우연히 원래 값이 다시 뽑힐 수 있어 '실제로 바뀐 개수'는 25보다 작다)
print(f"라벨 오염: 150개 중 {(y_dirty != yi).sum()}개가 실제로 바뀜")

# 오염된 라벨로 다시 분할. stratify 는 원본 yi 기준으로 (품종 비율 유지)
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(
    Xi, y_dirty, test_size=0.3, random_state=42, stratify=yi
)

print(f"\n{'max_depth':>10} | {'train':>7} | {'test':>7} | 격차")
print("-" * 42)
for d in [None, 5, 3, 2]:                        # 복잡한 모델 → 단순한 모델 순서
    t = DecisionTreeClassifier(max_depth=d, random_state=0).fit(Xd_tr, yd_tr)
    a = accuracy_score(yd_tr, t.predict(Xd_tr))  # train 정확도 (배운 데이터로 채점)
    b = accuracy_score(yd_te, t.predict(Xd_te))  # test 정확도 (처음 보는 데이터로 채점)
    print(f"{str(d):>10} | {a:>7.3f} | {b:>7.3f} | {a - b:+.3f}")

### 결과 읽는 법 — 이 표가 오늘의 핵심 중 하나다

```
 max_depth |   train |    test | 격차
      None |   1.000 |   0.689 | +0.311
         5 |   0.943 |   0.711 | +0.232
         3 |   0.886 |   0.800 | +0.086
         2 |   0.876 |   0.800 | +0.076
```

| | train | test | 무슨 일이 |
|---|---|---|---|
| `None` | **1.000** | **0.689** | 배운 건 100% 맞히는데 실전은 69%. **잘못된 라벨까지 외웠다** |
| `3` | 0.886 | **0.800** | train 은 더 낮은데 **test 는 더 높다** |

> 🔥 **`train 1.000` 짜리 모델이 가장 나쁜 모델이었다.**
> 깊이 제한이 없으면 트리는 데이터를 계속 쪼개서 **오염된 라벨까지 하나하나 외워 버린다.**
> 그게 `1.000` 의 정체다 — 실력이 아니라 **암기**이고, 처음 보는 데이터 앞에서 무너진다.
>
> 이걸 **과적합(overfitting)** 이라 한다. **train 점수가 아니라 train-test 격차**를 봐야
> 하는 이유이고, 애초에 데이터를 **train/test 로 나누는 이유**다.
> 나누지 않았다면 `1.000` 을 보고 "완벽한 모델!"이라고 발표했을 것이다.

> 🧠 **딥러닝의 train loss ↓ / val loss ↑ 갈라지는 그 그래프**와 같은 현상이다.
> 여기서는 epoch 축이 없으니 곡선 대신 **두 숫자의 차이**로 본다.

## 4. 🔥 함정 — 정확도가 거짓말할 때 (교안 3.8)

**오늘 가장 중요한 절이다.**

찾아야 할 대상이 **아주 드문** 문제가 있다 — 불량품 검출(1%), 희귀병 진단, 이상거래 탐지.
이런 **불균형 데이터**에서는 정확도가 거짓말을 한다.

확인 방법: **아무것도 학습하지 않는 모델**을 만들어서 점수를 재 본다.
`DummyClassifier(strategy="most_frequent")` 는 **무조건 다수 클래스**로만 찍는다.

In [ ]:
from sklearn.dummy import DummyClassifier          # 학습을 안 하는 '가짜' 모델 (기준선 확인용)
from sklearn.metrics import f1_score, recall_score

rng2 = np.random.default_rng(7)                  # 셀마다 독립 시드 → 실행 순서 무관
Xb = rng2.normal(size=(1000, 5))                 # 입력 1000개 (내용은 중요하지 않다)
yb = (rng2.random(1000) < 0.02).astype(int)      # 2% 확률로만 1(양성) → 희귀 사건

Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xb, yb, test_size=0.3, random_state=0)

# strategy="most_frequent": 학습 데이터에서 가장 많은 클래스로만 답한다 (여기선 항상 0)
dummy = DummyClassifier(strategy="most_frequent").fit(Xb_tr, yb_tr)
dp = dummy.predict(Xb_te)                        # 예측 결과: 전부 0

print(f"양성 비율: {yb.mean():.1%}  (불량품·질병·이상거래 같은 희귀 사건)")
print("\n'전부 정상'이라고만 찍는 모델의 성적표:")
print(f"  accuracy = {accuracy_score(yb_te, dp):.3f}   ← 훌륭해 보인다")
# recall: 진짜 양성 중 몇 개나 찾아냈나 (zero_division=0 은 0으로 나누는 경고 방지)
print(f"  recall   = {recall_score(yb_te, dp, zero_division=0):.3f}   ← 찾아야 할 걸 하나도 못 잡았다")
print(f"  f1       = {f1_score(yb_te, dp, zero_division=0):.3f}")

### 결과 읽는 법 — 98점짜리 무능한 모델

```
양성 비율: 2.6%
  accuracy = 0.983
  recall   = 0.000
  f1       = 0.000
```

**`정확도 98.3%`** — 학습을 **한 글자도** 안 한 모델이다.
양성이 2.6%뿐이니 **전부 "정상"이라고만 찍어도 97% 이상은 자동으로 맞는다.**

그런데 **`recall 0.000`** — **정작 찾아야 할 것을 하나도 못 잡았다.**
불량품 검출기라면 불량을 전부 통과시킨 것이고, 질병 진단이라면 환자를 전부 놓친 것이다.
**쓸모가 0인데 성적표는 98점이다.**

### 지표는 문제에 따라 고른다

| 상황 | 봐야 할 지표 | 뜻 |
|---|---|---|
| 놓치면 큰일 (암 진단, 불량 검출) | **recall(재현율)** | 진짜 양성 중 **몇 개나 찾아냈나** |
| 잘못 잡으면 곤란 (스팸 분류) | **precision(정밀도)** | 양성이라 한 것 중 **몇 개가 진짜였나** |
| 둘 다 중요 | **f1** | 두 지표의 조화평균 |

> ⚠️ **에이전트에게 지표를 지정해 주지 않으면 대개 accuracy 를 보고한다.**
> *"정확도 98% 나왔습니다"* 라는 보고를 그대로 믿으면 안 되는 이유다 —
> **무엇을 맞혀야 하는 문제인지는 사람이 정해서 알려 줘야 한다.**

## 5. 스윕 — 어느 깊이가 좋은가 (교안 3.9)

3절의 오염된 데이터를 그대로 쓴다. `max_depth` 를 바꿔 가며 test 점수만 본다.
이런 **반복이야말로 에이전트에게 맡길 일**이다 — 사람은 *어떤 축을 볼지*만 정한다.

In [ ]:
print(f"{'max_depth':>10} | {'test acc':>8}")
print("-" * 23)
for d in [1, 2, 3, 5, None]:                     # 단순 → 복잡 순서로
    t = DecisionTreeClassifier(max_depth=d, random_state=0).fit(Xd_tr, yd_tr)
    print(f"{str(d):>10} | {accuracy_score(yd_te, t.predict(Xd_te)):>8.3f}")

### 결과 읽는 법 — **가운데가 가장 높은 산 모양**

```
 max_depth | test acc
         1 |    0.556
         2 |    0.800
         3 |    0.800
         5 |    0.711
      None |    0.689
```

- `1`: **너무 단순해서** 못 배웠다 → **과소적합(underfitting)**
- `2~3`: **딱 좋다**
- `5~None`: **너무 복잡해서** 외워 버렸다 → **과적합**(3절)

> 🔥 **모델은 크다고 좋은 게 아니라 "알맞아야" 좋다.** 이 산 모양이 하이퍼파라미터
> 조정의 본질이다. 그리고 **꼭대기가 어디인지는 돌려 봐야만 안다** — 그래서 반복이
> 필요하고, 그 반복이 위임 대상이다.

> ⚠️ test 가 45개뿐이라 `0.800` 과 `0.778` 의 차이는 **1건**이다.
> 그 차이로 우열을 논하면 안 된다.

---

## 정리 — 오늘 배운 '읽는 법' 네 가지

| | 배운 것 | 근거 숫자 |
|---|---|---|
| 1 | **baseline 과 비교하라** | `R² 0.477` = 평균 찍기보다 조금 나은 정도 |
| 2 | **혼동행렬을 보라** | 정확도 뒤에 *versicolor ↔ virginica* 혼동이 숨어 있었다 |
| 3 | **train 점수를 믿지 마라** | `train 1.000` 짜리가 가장 나빴다(`test 0.689`) |
| 4 | **지표를 문제에 맞게 골라라** | `accuracy 0.983` / `recall 0.000` |

**모델을 만드는 데는 지시 한 줄이면 됐다. 시간의 대부분은 나온 숫자를 읽는 데 썼다.**
그게 요점이다 — **AI 에게 일을 시키는 능력과, 그 결과를 판단하는 능력은 다른 능력이다.**